# 01 — Generate Synthetic Data

**Project:** AdIntel AI  
**Milestone:** 1 — Synthetic Ads Dataset

Notebook ini digunakan untuk menjalankan script generator dan melakukan quick inspection terhadap output CSV.

> Asumsi: file `scripts/generate_synthetic_data.py` sudah tersedia di root project.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "generate_synthetic_data.py"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Script path:", SCRIPT_PATH)


## 1. Check Script Availability


In [ ]:
assert SCRIPT_PATH.exists(), f"Generator script tidak ditemukan: {SCRIPT_PATH}"
print("Generator script found.")


## 2. Run Synthetic Data Generator

Cell ini akan menjalankan script generator dari notebook. Kalau kamu sudah menjalankan script dari terminal, cell ini tetap aman dijalankan ulang karena output CSV akan di-overwrite.


In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

assert result.returncode == 0, "Synthetic data generation failed."


## 3. List Generated Files


In [ ]:
expected_files = [
    "advertisers.csv",
    "campaigns.csv",
    "ad_groups.csv",
    "creatives.csv",
    "placements.csv",
    "inventory.csv",
    "markets.csv",
    "devices.csv",
    "daily_ad_performance.csv",
    "video_performance.csv",
    "conversion_events.csv",
    "billing_revenue.csv",
    "data_quality_logs.csv",
]

generated_files = sorted([p.name for p in DATA_DIR.glob("*.csv")])

print("Generated files:")
for file in generated_files:
    print("-", file)

missing_files = set(expected_files) - set(generated_files)
assert not missing_files, f"Missing generated files: {missing_files}"


## 4. Row Count Summary


In [ ]:
row_counts = []

for file in expected_files:
    path = DATA_DIR / file
    df = pd.read_csv(path)
    row_counts.append({
        "table": file.replace(".csv", ""),
        "rows": len(df),
        "columns": df.shape[1]
    })

row_counts_df = pd.DataFrame(row_counts).sort_values("table").reset_index(drop=True)
row_counts_df


## 5. Preview Key Tables


In [ ]:
advertisers = pd.read_csv(DATA_DIR / "advertisers.csv")
campaigns = pd.read_csv(DATA_DIR / "campaigns.csv")
ad_groups = pd.read_csv(DATA_DIR / "ad_groups.csv")
creatives = pd.read_csv(DATA_DIR / "creatives.csv")
placements = pd.read_csv(DATA_DIR / "placements.csv")
daily_perf = pd.read_csv(DATA_DIR / "daily_ad_performance.csv")

display(advertisers.head())
display(campaigns.head())
display(ad_groups.head())
display(creatives.head())
display(placements.head())
display(daily_perf.head())


## 6. Basic Date Coverage


In [ ]:
daily_perf["date"] = pd.to_datetime(daily_perf["date"])

coverage = pd.DataFrame({
    "min_date": [daily_perf["date"].min()],
    "max_date": [daily_perf["date"].max()],
    "unique_days": [daily_perf["date"].nunique()],
    "total_rows": [len(daily_perf)]
})

coverage


## 7. Output

Kalau semua cell berhasil, dataset synthetic sudah siap divalidasi lebih lanjut di notebook:

`02_validate_metrics.ipynb`
